# Radial Velocity Variability

In [ ]:
from pathlib import Path
import sys
import pickle

import numpy as np

import pandas as pd
import h5py

import matplotlib.pyplot as plt
%matplotlib widget

sys.path.append('/nexus/posix0/MIA-astro-env/hxr/bepennell/emission_line_stars/HotStarsBOSS/code/')
sys.path.append('../code/')
from spec_utils import load_first_spectrum

# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)


In [ ]:
# set random seed
rng = np.random.default_rng(17)

In [ ]:
CONFIG = {
    'project_root': '/nexus/posix0/MIA-astro-env/hxr/bepennell/emission_line_stars/HotStarsBOSS',
    'merged_star_file': '/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/info/allStar-v6_2_1_merged_params.parquet', 
    'merged_spectra_file': '/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/info/spAll-v6_2_1_merged_params.parquet',
    'bossnet_file': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/bossnet/astraAllStarBossNet_filtered.parquet'),
    'raw_cache_dir': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/boss_v6_2_1_cache'),
    'normalised_dir': Path('/nexus/posix0/MIA-astro-env/hxr/shared/FEROS_data/BOSS_data/boss_v6_2_1_normalised'),
}

# CONFIG['output_dir'].mkdir(parents=True, exist_ok=True)
# CONFIG['figures_dir'].mkdir(parents=True, exist_ok=True)

## Load catalogues and merge data

In [ ]:
og_df_stars = pd.read_parquet(CONFIG['merged_star_file']) #original star file that won't go through filtering

df_stars = pd.read_parquet(CONFIG['merged_star_file']) 
df_spectra = pd.read_parquet(CONFIG['merged_spectra_file'])

for col in ['GAIA_ID', 'SDSS_ID', 'FIELD', 'MJD']:
    if col in df_spectra.columns:
        df_spectra[col] = pd.to_numeric(df_spectra[col], errors='coerce').astype('Int64')
    if col in df_stars.columns:
        df_stars[col] = pd.to_numeric(df_stars[col], errors='coerce').astype('Int64')

print('Merged spectra rows:', len(df_spectra))
print('Merged star rows:', len(df_stars))
df_spectra.head(5)

In [ ]:
# data frame of stars with GAIA_ID, SDSS_ID, and Teff_fit columns
df_stars[['GAIA_ID', 'SDSS_ID','Teff_fit']].head(5)

example_id = 54381035

df_spectra.loc[df_spectra['SDSS_ID'] == example_id][['GAIA_ID', 'SDSS_ID', 'FIELD', 'MJD',]]



In [ ]:

for key in df_stars.keys():
    print(key, df_stars[key].dtype)

fig = plt.figure(figsize=(10, 6)   )
plt.hist(df_stars['Teff_fit'], bins=100, histtype='step', color='black')
plt.xlabel('EWobs_Halpha_mean')
plt.ylabel('Frequency')
plt.title('Distribution of EWobs_Halpha_mean')
plt.show()


In [ ]:
# -----------------------------
# Quality cuts for stars: choose 
# -----------------------------

mask_quality_stars = (df_stars['is_quality_star'] == True)
mask_hot_stars = (df_stars['Teff_fit'] >= 15000) # make a histo to see distribution
mask_no_emission = df_stars['EWobs_Halpha_mean'] >0.

cold_star_inds = og_df_stars.loc[mask_quality_stars & ~mask_hot_stars].index
emission_star_inds = og_df_stars.loc[mask_quality_stars & ~mask_no_emission].index

print(f'Quality filtering for stars: kept {len(df_stars.loc[mask_quality_stars & mask_hot_stars & mask_no_emission])} / {len(df_stars)} rows')
print(f"Stars with Teff_fit >= 10000: {len(df_stars.loc[mask_hot_stars])} / {len(df_stars)} rows")
print(f"Stars with EWobs_Halpha_mean > 0: {len(df_stars.loc[mask_no_emission])} / {len(df_stars)} rows")
print(f'Quality filtering for stars: kept {len(df_stars.loc[mask_quality_stars])} / {len(df_stars)} rows')

df_stars = df_stars.loc[mask_quality_stars & mask_hot_stars & mask_no_emission]

mask_stars = np.isin(df_spectra['SDSS_ID'], df_stars.loc[mask_quality_stars, 'sdss_id'])
mask_quality_spectra = (df_spectra['is_quality_spectrum'] == True)

print(f'Quality filtering for spectra: kept {len(df_spectra.loc[mask_quality_spectra&mask_stars])} / {len(df_spectra)} rows')
df_spectra = df_spectra.loc[mask_quality_spectra&mask_stars]

### Example plots from cleaned sample (hot and no emission)

In [ ]:
df_stars[df_stars['Teff_fit']>28000]#.index #get some indices that make cuts

In [ ]:
indices = [35,121,171,184, 934]

for ind in indices:
    example_spec_file = df_stars.loc[ind, 'SPEC_FILE']

    wave, flux, ivar = load_first_spectrum(
        df_stars,
        raw_cache_dir=CONFIG['raw_cache_dir'],
        spectrum_file=example_spec_file,
        normalised=True,
        normalised_dir=CONFIG['normalised_dir'],
        )

    fig = plt.figure(figsize=[10, 4])
    plt.fill_between(wave, flux - ivar**-0.5, flux + ivar**-0.5, alpha=0.3)
    plt.errorbar(wave, flux, yerr=ivar**-0.5, label=df_stars.loc[ind, 'GAIA_ID'], linestyle='None', marker='None')
    plt.xlabel('Wavelength')
    plt.ylabel('Flux')
    plt.title("example from cleaned sample (hot, no emission)")
    plt.legend()
    plt.show()

### Cool star (removed)

In [ ]:
cold_star_inds = cold_star_inds[:4] 
for ind in cold_star_inds:
    print(ind)
    example_spec_file = og_df_stars.loc[ind, 'SPEC_FILE']

    wave, flux, ivar = load_first_spectrum(
        og_df_stars, 
        raw_cache_dir=CONFIG['raw_cache_dir'],
        spectrum_file=example_spec_file,
        normalised=True,
        normalised_dir=CONFIG['normalised_dir'],
        )

    fig = plt.figure(figsize=[10, 4])
    plt.fill_between(wave, flux - ivar**-0.5, flux + ivar**-0.5, alpha=0.3)
    plt.errorbar(wave, flux, yerr=ivar**-0.5, label=og_df_stars.loc[ind, 'GAIA_ID'], linestyle='None', marker='None')
    plt.xlabel('Wavelength')
    plt.ylabel('Flux')
    plt.title("example of cold stars")
    plt.legend()
    plt.show()
    

### Halpha emission (removed)

In [ ]:
emission_star_inds = emission_star_inds[:4] 
for ind in emission_star_inds:
    print(ind)
    example_spec_file = og_df_stars.loc[ind, 'SPEC_FILE']

    wave, flux, ivar = load_first_spectrum(
        og_df_stars, 
        raw_cache_dir=CONFIG['raw_cache_dir'],
        spectrum_file=example_spec_file,
        normalised=True,
        normalised_dir=CONFIG['normalised_dir'],
        )

    fig = plt.figure(figsize=[10, 4])
    plt.fill_between(wave, flux - ivar**-0.5, flux + ivar**-0.5, alpha=0.3)
    plt.errorbar(wave, flux, yerr=ivar**-0.5, label=og_df_stars.loc[ind, 'GAIA_ID'], linestyle='None', marker='None')
    plt.xlabel('Wavelength')
    plt.ylabel('Flux')
    plt.title("example of emission-line stars")
    plt.legend()
    plt.show()
    

### saving filtered stars and filtered spectra as fits files

In [ ]:
stars_fn = "stars_2026-06-27.parquet"
spectra_fn = "spectra_2026-06-27.parquet"
df_stars.to_parquet(stars_fn)
df_spectra.to_parquet(spectra_fn)

### saving sample of 2000 spectra

In [ ]:
# first let's choose 2000 stars
hot_stars_subsample = pd.read_parquet(stars_fn)
subset_index = np.sort(rng.choice(np.arange(len(hot_stars_subsample)), size=2000, replace=False))
hot_stars_subsample = hot_stars_subsample.iloc[subset_index]
print(len(hot_stars_subsample), hot_stars_subsample)

In [ ]:
print(hot_stars_subsample["SPEC_FILE"])

In [ ]:
print(df_spectra.columns.to_list())

In [ ]:
# add a column to record our success or failure on downloading?!
hot_stars_subsample.insert(loc = 1, column = "successfully_read_file", value = False)

In [ ]:
normalized_dir=CONFIG['normalised_dir']

# with zipfile.ZipFile('spectra_sample_2000.zip', 'w') as zipf:
#     for spec_file in spec_files:
#         full_path = normalized_dir / spec_file
#         zipf.write(full_path, arcname=os.path.basename(spec_file)

N = len(hot_stars_subsample)
M = None
for ii, (_, row) in enumerate(hot_stars_subsample[["GAIA_ID", "SPEC_FILE"]].iterrows()):
        
    spec_file = row["SPEC_FILE"]
    plate = spec_file.split('-')[1] 
    key = spec_file.replace('.fits', '')  
    with h5py.File(normalized_dir / f'{plate}.h5', 'r') as f: 
            
            flux1 = f[key]['FLUX'][:]        
            loglam1 = f[key]['LOGLAM'][:]   
            ivar1 = f[key]['IVAR'][:]        
            continuum1 = f[key]['CONTINUUM'][:]

            if M is None:
                M = len(flux1)
                flux = np.zeros((N, M))
                loglam = loglam1.copy()
                ivar = np.zeros((N, M))
                continuum = np.zeros((N, M))
            assert len(flux1) == M
            assert np.allclose(loglam1, loglam)

            flux[ii] = flux1
            ivar[ii] = ivar1
            continuum[ii] = continuum1
            hot_stars_subsample["successfully_read_file"] = True

print(flux.shape, loglam.shape, ivar.shape, continuum.shape)

In [ ]:
#save hot stars sample

file_prefix = "hot_stars_subsample_2026-06-27"
hot_stars_subsample.to_parquet(file_prefix + "_table.parquet")

In [ ]:
print(file_prefix)

In [ ]:
# how many failures did we have?
print(hot_stars_subsample[hot_stars_subsample["successfully_read_file"] == False])

In [ ]:
# spot check
plt.figure()
plt.plot(flux[1995])
plt.show()

In [ ]:
# write a pickle file

with open(file_prefix + "_data.pkl", "wb") as file:
    pickle.dump((flux, loglam, ivar, continuum), file)